In [16]:
import warnings
warnings.filterwarnings("ignore")

In [17]:
import google.auth
from google.adk.telemetry import google_cloud
from google.adk.telemetry.setup import maybe_set_otel_providers

# Resolve credentials and project ID explicitly
credentials, project_id = google.auth.default()
print(f"Project ID: {project_id}")

# Build OTel resource with the project ID so Cloud Trace knows where to export
resource = google_cloud.get_gcp_resource(project_id=project_id)

# Get GCP exporters configuration
hooks = google_cloud.get_gcp_exporters(
    enable_cloud_tracing=True,
    google_auth=(credentials, project_id),
)

# Initialize and set global OTel providers
maybe_set_otel_providers(otel_hooks_to_setup=[hooks], otel_resource=resource)

print("✅ Cloud Trace initialized")

Overriding of current TracerProvider is not allowed


Project ID: dev-mq-tech-transfer
✅ Cloud Trace initialized


In [18]:
from dotenv import load_dotenv


import asyncio
import os

from google.genai import types
from google.adk.agents.llm_agent import LlmAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.artifacts.in_memory_artifact_service import InMemoryArtifactService # Optional
from google.adk.planners import BasePlanner, BuiltInPlanner, PlanReActPlanner
from google.adk.models import LlmRequest

from google.genai.types import ThinkingConfig
from google.genai.types import GenerateContentConfig

import datetime
from zoneinfo import ZoneInfo

APP_NAME = "weather_app"
USER_ID = "1234"
SESSION_ID = "session1234"

def get_weather(city: str) -> dict:
    """Retrieves the current weather report for a specified city.

    Args:
        city (str): The name of the city for which to retrieve the weather report.

    Returns:
        dict: status and result or error msg.
    """
    if city.lower() == "new york":
        return {
            "status": "success",
            "report": (
                "The weather in New York is sunny with a temperature of 25 degrees"
                " Celsius (77 degrees Fahrenheit)."
            ),
        }
    else:
        return {
            "status": "error",
            "error_message": f"Weather information for '{city}' is not available.",
        }


def get_current_time(city: str) -> dict:
    """Returns the current time in a specified city.

    Args:
        city (str): The name of the city for which to retrieve the current time.

    Returns:
        dict: status and result or error msg.
    """

    if city.lower() == "new york":
        tz_identifier = "America/New_York"
    else:
        return {
            "status": "error",
            "error_message": (
                f"Sorry, I don't have timezone information for {city}."
            ),
        }

    tz = ZoneInfo(tz_identifier)
    now = datetime.datetime.now(tz)
    report = (
        f'The current time in {city} is {now.strftime("%Y-%m-%d %H:%M:%S %Z%z")}'
    )
    return {"status": "success", "report": report}

# Step 1: Create a ThinkingConfig
thinking_config = ThinkingConfig(
    include_thoughts=True,   # Ask the model to include its thoughts in the response
    thinking_budget=256      # Limit the 'thinking' to 256 tokens (adjust as needed)
)
print("ThinkingConfig:", thinking_config)

# Step 2: Instantiate BuiltInPlanner
planner = BuiltInPlanner(
    thinking_config=thinking_config
)
print("BuiltInPlanner created.")

# Step 3: Wrap the planner in an LlmAgent
agent = LlmAgent(
    model="gemini-2.5-flash",
    name="weather_and_time_agent",
    instruction="You are an agent that returns time and weather",
    planner=planner,
    tools=[get_weather, get_current_time]
)

# Session and Runner
session_service = InMemorySessionService()
session = await session_service.create_session(app_name=APP_NAME, user_id=USER_ID, session_id=SESSION_ID)
runner = Runner(agent=agent, app_name=APP_NAME, session_service=session_service)

# Agent Interaction
def call_agent(query):
    content = types.Content(role='user', parts=[types.Part(text=query)])
    events = runner.run(user_id=USER_ID, session_id=SESSION_ID, new_message=content)

    for event in events:
        print(f"\nDEBUG EVENT: {event}\n")
        if event.is_final_response() and event.content:
            final_answer = event.content.parts[0].text.strip()
            print("\n🟢 FINAL ANSWER\n", final_answer, "\n")

call_agent("If it's raining in New York right now, what is the current temperature?")

ThinkingConfig: include_thoughts=True thinking_budget=256 thinking_level=None
BuiltInPlanner created.

DEBUG EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Okay, so I'm trying to figure out what the user wants. They've asked two distinct things, really: they want to know if it's currently raining in New York, and they also want to know the current temperature there.

My first thought is to look for a tool that can give me weather information. The `get_weather` tool seems like a perfect fit for both of these inquiries. It clearly takes a `city` parameter, which in this case is "New York."

So, I'm ready to call the `get_weather` tool with "New York" as the city.

However, here's where I hit a bit of a snag. The `get_weather` tool *can* indeed retrieve weather reports. But the user's first question is conditional: " *If* it's raining..." The tool's description says it "Retrieves the current weather report," which is great, but it doesn't explic

Exception while exporting Span.
Traceback (most recent call last):
  File "c:\Users\L137860\OneDrive - Eli Lilly and Company\Playground\gcp_trace_exp\gcp_trace_test\.venv\Lib\site-packages\opentelemetry\sdk\_shared_internal\__init__.py", line 194, in _export
    self._exporter.export(
  File "c:\Users\L137860\OneDrive - Eli Lilly and Company\Playground\gcp_trace_exp\gcp_trace_test\.venv\Lib\site-packages\opentelemetry\exporter\otlp\proto\http\trace_exporter\__init__.py", line 185, in export
    resp = self._export(serialized_data, deadline_sec - time())
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\L137860\OneDrive - Eli Lilly and Company\Playground\gcp_trace_exp\gcp_trace_test\.venv\Lib\site-packages\opentelemetry\exporter\otlp\proto\http\trace_exporter\__init__.py", line 157, in _export
    resp = self._session.post(
           ^^^^^^^^^^^^^^^^^^^
  File "c:\Users\L137860\OneDrive - Eli Lilly and Company\Playground\gcp_trace_exp\gcp_trace_test\.venv